# Burnback simulation pipeline

Continuation of [Sim](Sim.ipynb)

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
from duckdb import sql as sqldf


from Rockets.Simulation.Plots import dots_and_arrows, animate, Web, Perf, casing, \
                                     Log, multiplot ,save_fig, add_plot
from Rockets.Simulation.Plots import preproc_curves_data, preproc_sim_data, preproc_log_data 
from Rockets.Simulation.Simulation import Lagrangian
from Rockets.Simulation.Pipelines import SimPipeline
from Rockets.Superformula.Formulas import formula1, formula2
from Rockets.Geometry import cart2pol, pol2cart, rad2deg, normals, magn
from Rockets.utils import clip, shape, pick, parse_sf_params
from Benchmarking.utils import describe, JsonObjectEncoder, TrackWithID

In [ ]:
simParams = {
"d": .05,
"steps": 70,
"n": 1000,
"window_size": 70,
"hull_radius": 4,
"interp": 'manydumb'
}


dest = "../../../data/Generated-dbg/"

## Simulation run

[v] Clear Caustics  
[v] Intersection point as replacement for caustic  
[v] Intersection window processes the whole closed circle  
[v] Fill Rarefactions (primitive interpolation)  
[ ] Rarefactions - better interpolation  
[ ] Interpolate multiple point in between rarefactions  
[v] Points stop advancing when front reaches casing  
[ ] Points stop exactly on the casing  
[v] Separation of burning regions  
[ ] Handling cusps - regions that separate entirely from front [?]  
[v] Harvester - gathers data during sim for dbg and visualization  
[v] Interactive plot - animation of burning front  
[ ] Dots and arrows plot - zoom into data of specific steps for debugging  
[ ] Compare results and timings with Kang simulation.  

### Multiple shapes runs 

In [ ]:

from pathlib import Path

patt = "*"
path = Path("../../../data/SFs")

paths = list(path.glob(patt +".png"))

tracks = {
    "episodes": {
        "sampling_type": "interval_episodes",
        "sampling_value": 1},
    "harvest": {
        "curves": {
            "varnames": ["self.I", "self.XY", "self.A", "self.IsNew",
                        "self.SimStep", "self.N", "E", "self.d", "filt"],
            "on_size_mismatch": "error"}
            }
}
                                    #"I1","X1", "Y1", circ, , "isNew" 




cases = [{"config":{
              "file":
                  {"path": p.resolve().as_posix() , 
                   "name": p.name, 
                   "stem": p.stem}, 
              "SFparams": parse_sf_params(p.stem),
              "simulation": simParams,},
          "ID": 
              {"case_signature": p.stem},
          "tracks": tracks
          } for p in paths ]
   
cases[0]


In [ ]:
import warnings

# Ignore all FutureWarnings globally
warnings.filterwarnings("ignore", category=FutureWarning)


from Benchmarking.Benchmarking import Bench
from Benchmarking.utils import isDebugging

dataroot = "/home/michael/Studies/DSlab/Capstone/data/"

bench = Bench(benchmarks_root=dataroot + "benchmarks",
              output_root= dataroot + "output",
              folder="20260805/1727",
              onerror = "fail")


### Get Logs data

In [ ]:
TELE_log = TrackWithID([bench.TELE.CAse], 'log')[:-2]  #['tracks']['log']['data']

# 2 last log entries in the TELE are "loading experiment from..." 
# and "loaded 33 files from.." from just above when loading experiment
# they zoom out the plot too much, so I drop them

In [ ]:
describe(TELE_log)

In [ ]:
TELE_log

In [ ]:
aCAse = bench.DONEcases[20]
aCAse.case_signature

In [ ]:
describe(aCAse)

In [ ]:
describe(bench.DONEcases)

In [ ]:
aCaselog = TrackWithID([aCAse], 'log')
aCaselog

In [ ]:
describe(aCaselog)

In [ ]:
Caseslog = TrackWithID(bench.DONEcases, 'log')
describe(Caseslog)

In [ ]:
wat, tf = preproc_log_data(Caseslog)
wat

In [ ]:

Experiment_Log = TELE_log + Caseslog

In [ ]:
logdf, log_cmap = preproc_log_data(Experiment_Log)
logdf

In [ ]:
logdf = logdf.drop_duplicates('time', keep = 'last')
logdf

In [ ]:
Log(logdf, log_cmap, render_mode = 'SVG')

### Web burning and performance Plots

In [ ]:
tracks = aCAse.tracks
config = aCAse.config

simdf = preproc_sim_data(tracks['episodes']['data'])
curvesdf, curve_colors =  preproc_curves_data(tracks['harvest']['curves']['data'])
case_file = aCAse.case_filename()



In [ ]:
W = Web(curvesDF=curvesdf, hull_radius=  config["simulation"]["hull_radius"],render_mode="SVG", width = 500,) #   ,
W

In [ ]:
P = Perf(simdf, render_mode="SVG")

In [ ]:
A = animate(curvesdf, width = 600, height = 600,
        hull_radius = config["simulation"]["hull_radius"],
        color_map=curve_colors)
A

In [ ]:
MP = multiplot(1,2)

add_plot(MP, W, row=1, col=1)

add_plot(MP, P, row=1, col=2)

# add_plot(MP, animate(curvesdf, hull_radius = config["simulation"]["hull_radius"], color_map=curve_colors), row = 2, col = 1)

In [ ]:

simppl = SimPipeline()

#case_template = simppl.case_template()
#case_template["tracks"].pop('profile')

bench.configure(simppl)

bench.set_cases(cases[2:9])
# bench.unfurl_grid(case_template, chosengrid)

bench.run_experiments()


In [ ]:
bench.pipeline.current_case

In [ ]:
bench.DONEcases.__len__()

In [ ]:
tree = bench.DONEcases[5]#['tracks']#['episodes']#['data']


In [ ]:
import importlib
import Benchmarking
importlib.reload(Benchmarking.utils)
from Benchmarking.utils import describe

In [ ]:
wat = describe(tree)

In [ ]:
tree['tracks']['episodes']['data']

In [ ]:


def simPipeline(Case, dest):

    profile = formula1(**Case["SFparams"])

    simulation = Case["simulation"] 

    SIM = Lagrangian(profile, **simulation)

    # Shape(SIM.R, SIM.T)
    SIM.run(simulation["steps"])
    print("Simulation completed.")

    
    # HSintrsctns = SIM.HSintersections.results()
    # HSsimdata = SIM.HSsim.results()


    WebAndPerf(SIM, dest + Case['stem'] + ".html")

    return SIM
    # dfSim = pd.DataFrame(HSsimdata)
    # dfSim   
    # px.line(dfSim, x = 'SimStep', y = 'C')

In [ ]:
SIM = simPipeline(cases[0], dest)

In [ ]:
animdf = animate(SIM)

In [ ]:
animdf

In [ ]:
animdf.X.notna()

In [ ]:
px.histogram(animdf[animdf.X.notna()].stat)

In [ ]:
WebAndPerf(SIM)

In [ ]:


# StepsFilter = (11,12) 
# StepsFilter = (3,4) 
StepsFilter = (4,5,6)
StepsFilter = (31,32,33)

filtr = {"SimStep": StepsFilter}


In [ ]:

dots_and_arrows(SIM, filtr = filtr )

In [ ]:
interp = 'slerp'
SIM2 = SimPipeline(cases[0], dest)

In [ ]:
dots_and_arrows(SIM2, filtr)

In [ ]:
interp = 'manydumb'
d = 0.01
SIM3 = SimPipeline(cases[0], dest)
dots_and_arrows(SIM3, filtr)

In [ ]:
animate(SIM)

In [ ]:
q = """ --
select  *, cast(IsNew as bool)  as isNeww 
from hsdf
where SimStep in (2,3) -- and (I < 10 or I > 700)
-- group by item , tbl
--order by non_null_values desc
"""

wat = sqldf(q).df()

wat